In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, brier_score_loss
from sklearn.feature_selection import SelectFromModel, SelectKBest, f_classif, mutual_info_classif, VarianceThreshold, RFECV
from PersonaClassifier import My_training, Dataset, generate_cal_result
from utils.Visualization import generate_cm, generate_auroc, display_auroc
from utils.Models import MLPWrapper, MLP, BiLSTMClassifier, IdentityEstimator
from sklearn.metrics import make_scorer, accuracy_score
print(torch.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [2]:
demo= 50000
my_train = My_training(model_list=['mlp'], emb_model='roberta-base', demo=demo)
train_set = Dataset('./processed_data/2-splits/pandora_train_val_v2.csv', my_train.emb_model, my_train.traits, my_train.demo) 
selected_features = {}

/home/jmaharja/anaconda3/envs/gpu/lib/python3.8/site-packages/transformers/utils/generic.py:260: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.weight', 'roberta.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
for target_col in my_train.traits:
    print(f'{10*"-"} {target_col} {10*"-"}')
    X, y, selected_features[target_col] = my_train.prepare_dataset(train_set.X, train_set.contextual_emb, train_set.Y[[target_col]], [])
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, stratify=y, shuffle=True, random_state=42)
    param_grid = {
        # 'hidden_size': [128, 256],
        'lr': [0.001, 0.0001],
        # 'epochs': [10, 16],
        # 'batch_size': [8, 16],
        # 'dropout_rate' :[0.3, 0.5]
    }
    scoring = ['accuracy']

    # Wrap the model
    model = BiLSTMClassifier(input_dim=X.shape[1], hidden_dim=128, output_dim=1, num_layers=2, bidirectional=True, do_attention=True, dropout_rate=0.5)
    # model = MLP(input_size=X_train.shape[1], hidden_size=128, output_size=1, dropout_rate=0.3)
    pytorch_model = MLPWrapper(model, False, epochs=10, batch_size=16, lr=0.001)

    # Perform Grid Search
    # RandomizedSearchCV
    grid_search = GridSearchCV(estimator=pytorch_model, param_grid=param_grid, cv=3, refit='accuracy', scoring='accuracy', verbose=1)
    grid_search.fit(X_train, y_train)

    # Best parameters and model
    print("Best Parameters:", grid_search.best_params_)
    # print("Best Model:", grid_search.best_estimator_)

    # Evaluate on test data
    best_model = grid_search.best_estimator_
    print("Test Accuracy:", best_model.score(X_test, y_test))

---------- cOPN ----------
Fitting 3 folds for each of 2 candidates, totalling 6 fits
Best Parameters: {'lr': 0.001}
Test Accuracy: 0.6422
---------- cCON ----------
Fitting 3 folds for each of 2 candidates, totalling 6 fits
Best Parameters: {'lr': 0.001}
Test Accuracy: 0.6544
---------- cEXT ----------
Fitting 3 folds for each of 2 candidates, totalling 6 fits


In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from skorch import NeuralNetClassifier

bilstm = BiLSTMClassifier(input_dim=X.shape[1], hidden_dim=128, output_dim=1, num_layers=2, bidirectional=True, do_attention=True, dropout_rate=0.5)

# create model with skorch
model = NeuralNetClassifier(
    bilstm,
    criterion=nn.BCELoss,
    optimizer=optim.Adam,
    verbose=False
)

# define the grid search parameters
param_grid = {
    'batch_size': [8,16],
    'max_epochs': [10, 20]
}
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3)
grid_result = grid.fit(X, y)

# summarize results
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, stdev, param))

In [ ]:
demo= 10000
my_train = My_training(model_list=['mlp'], emb_model='roberta-base', demo=demo)
train_set = Dataset('./processed_data/2-splits/pandora_train_val_v2.csv', my_train.emb_model, my_train.traits, my_train.demo) 
for target_col in my_train.traits:
    print(target_col)
    X, y, _ = my_train.prepare_dataset(train_set.X, train_set.contextual_emb, train_set.Y[[target_col]], [])
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, shuffle=True, random_state=42)
    scoring = {'accuracy':  make_scorer(accuracy_score)}
    # scoring = ['accuracy', 'precision']
    # scoring = {'accuracy': 'accuracy', 'prec': 'precision', 'roc_auc':'roc_auc'}

    param_grid = {
        # 'hidden_size': [128, 256],
        'lr': [0.01, 0.001],
        'epochs': [10, 16],
        'batch_size': [8, 16]
    }

    # Wrap the model
    model = BiLSTMClassifier(input_dim=X.shape[1], hidden_dim=128, output_dim=1, num_layers=2, bidirectional=True, do_attention=True, dropout_rate=0.3)
    # mlp = MLP(input_size=X_train.shape[1], hidden_size=128, output_size=1, dropout_rate=0.3)
    pytorch_model = MLPWrapper(model, False, epochs=10, batch_size=8, lr=0.01)

    # Perform Grid Search
    grid_search = GridSearchCV(estimator=pytorch_model, param_grid=param_grid, cv=3, scoring='roc_auc', verbose=1)
    grid_search.fit(X_train, y_train)

    # Best parameters and model
    print("Best Parameters:", grid_search.best_params_)
    print("Best Model:", grid_search.best_estimator_)

    # Evaluate on test data
    best_model = grid_search.best_estimator_
    print("Test Accuracy:", best_model.score(X_test, y_test))


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import GridSearchCV
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np

# Define PyTorch Model
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.softmax(x)
        return x

# Define sklearn-compatible wrapper
class PyTorchClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, input_size, hidden_size, output_size, lr=0.001, epochs=10):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.lr = lr
        self.epochs = epochs
        self.model = SimpleNN(input_size, hidden_size, output_size)
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.lr)

    def fit(self, X, y):
        self.model.train()
        X_tensor = torch.tensor(X, dtype=torch.float32)
        y_tensor = torch.tensor(y, dtype=torch.long)
        for _ in range(self.epochs):
            self.optimizer.zero_grad()
            outputs = self.model(X_tensor)
            loss = self.criterion(outputs, y_tensor)
            loss.backward()
            self.optimizer.step()
        return self

    def predict(self, X):
        self.model.eval()
        with torch.no_grad():
            X_tensor = torch.tensor(X, dtype=torch.float32)
            outputs = self.model(X_tensor)
            _, predicted = torch.max(outputs, 1)
        return predicted.numpy()

    def score(self, X, y):
        y_pred = self.predict(X)
        return accuracy_score(y, y_pred)

# Load dataset
data = load_iris()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define parameter grid
param_grid = {
    'hidden_size': [10, 20, 50],
    'lr': [0.01, 0.001],
    'epochs': [10, 50]
}
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall'
}

# Wrap the model
pytorch_model = PyTorchClassifier(input_size=X.shape[1], hidden_size=10, output_size=3)

# Perform Grid Search
grid_search = GridSearchCV(estimator=pytorch_model, param_grid=param_grid, cv=3, scoring='accuracy', verbose=1)
grid_search.fit(X_train, y_train)

# Best parameters and model
print("Best Parameters:", grid_search.best_params_)
print("Best Model:", grid_search.best_estimator_)

# Evaluate on test data
best_model = grid_search.best_estimator_
print("Test Accuracy:", best_model.score(X_test, y_test))


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Sample data
X = [[0, 0], [1, 1]]
y = [0, 1]

# Define the parameter grid
param_grid = {'C': [0.1, 1, 10], 'gamma': [0.1, 1, 10]}

# Create the estimator
svc = SVC()

# Define the scoring metrics
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall'
}

# Create the GridSearchCV object
grid_search = GridSearchCV(svc, param_grid, scoring=scoring, refit='accuracy')

# Fit the data
grid_search.fit(X, y)

# Access the results
print("Best parameters:", grid_search.best_params_)
print("Best accuracy score:", grid_search.best_score_)

# Access the scores for all metrics
results = grid_search.cv_results_
print("Accuracy scores:", results['mean_test_accuracy'])
print("Precision scores:", results['mean_test_precision'])
print("Recall scores:", results['mean_test_recall'])
